In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
api_key=os.getenv('PINCONE_API_KEY')

In [3]:
from langchain_community.retrievers import PineconeHybridSearchRetriever


C:\Users\DELL\AppData\Local\Temp\ipykernel_17280\3778597445.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import PineconeHybridSearchRetriever


In [4]:
from pinecone import Pinecone,ServerlessSpec
index_name="hybrid-search-langchain-pinecone"
pc=Pinecone(api_key=api_key)

#create the index
if index_name  not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='dotproduct',#sparse value supported
        spec=ServerlessSpec(cloud='aws',region='us-east-1'),
    )

In [5]:
index=pc.index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-w6dt3tq.svc.aped-4627-b74a.pinecone.io')

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
hf_api_key=os.getenv('HF_TOKEN')
embeddins=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddins

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder=BM25Encoder().default()
bm25_encoder

In [9]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

## tfidf values on these sentence
bm25_encoder.fit(sentences)

## store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

  0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddins,sparse_encoder=bm25_encoder,index=index)


In [12]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x00000231D628C9D0>, index=Index(host='https://hybrid-search-langchain-pinecone-w6dt3tq.svc.aped-4627-b74a.pinecone.io'))

In [14]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]
)

  0%|          | 0/1 [00:00<?, ?it/s]

TypeError: Index.upsert() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given